In [ ]:
import pandas as pd
import numpy as np
import joblib
import json
import pickle
from google.colab import drive

drive.mount('/content/drive')

with open("/content/drive/MyDrive/SmartRail/time_split_data.pkl", "rb") as f:
    data = pickle.load(f)

X_train_time = data["X_train_time"]
y_train_time = data["y_train_time"]
X_test_time = data["X_test_time"]
y_test_time = data["y_test_time"]

print(X_train_time.shape, X_test_time.shape)

Mounted at /content/drive
(1160000, 44) (299444, 44)


In [ ]:
!pip install optuna -q

import optuna
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

sample_idx = np.random.choice(len(X_train_time), 150000, replace=False)
X_train_sample = X_train_time.iloc[sample_idx]
y_train_sample = y_train_time.iloc[sample_idx]

scale_pos_weight = (y_train_sample==0).sum() / (y_train_sample==1).sum()

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 200),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "scale_pos_weight": scale_pos_weight,
        "random_state": 42,
        "eval_metric": "logloss"
    }
    model = XGBClassifier(**params)
    model.fit(X_train_sample, y_train_sample)
    preds = model.predict_proba(X_test_time)[:, 1]
    return roc_auc_score(y_test_time, preds)

print("Objective function ready")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 5.7 MB/s eta 0:00:00
Objective function ready


In [ ]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20, show_progress_bar=True)

print("Best AUC:", study.best_value)
print("Best params:", study.best_params)

[I 2026-09-21 19:57:58,301] A new study created in memory with name: no-name-4778ec5f-78fe-4c74-bab1-ee35f1a23ac2


  0%|          | 0/20 [00:00<?, ?it/s]

[I 2026-09-21 19:58:08,529] Trial 0 finished with value: 0.9999838306390378 and parameters: {'n_estimators': 62, 'max_depth': 4, 'learning_rate': 0.14028129935681807, 'subsample': 0.6030169454238589, 'colsample_bytree': 0.9697212862752244, 'min_child_weight': 8}. Best is trial 0 with value: 0.9999838306390378.
[I 2026-09-21 19:58:13,070] Trial 1 finished with value: 0.9998148351883134 and parameters: {'n_estimators': 122, 'max_depth': 3, 'learning_rate': 0.02313860310075372, 'subsample': 0.7241529181706615, 'colsample_bytree': 0.6249012106872417, 'min_child_weight': 1}. Best is trial 0 with value: 0.9999838306390378.
[I 2026-09-21 19:58:16,899] Trial 2 finished with value: 0.9998957510422682 and parameters: {'n_estimators': 71, 'max_depth': 6, 'learning_rate': 0.17814208561622782, 'subsample': 0.8251634577039729, 'colsample_bytree': 0.6830981953110036, 'min_child_weight': 1}. Best is trial 0 with value: 0.9999838306390378.
[I 2026-09-21 19:58:25,181] Trial 3 finished with value: 0.9984

In [ ]:
best_params = study.best_params
best_params["scale_pos_weight"] = (y_train_time==0).sum() / (y_train_time==1).sum()
best_params["random_state"] = 42
best_params["eval_metric"] = "logloss"

xgb_tuned = XGBClassifier(**best_params)
xgb_tuned.fit(X_train_time, y_train_time)

auc_tuned = roc_auc_score(y_test_time, xgb_tuned.predict_proba(X_test_time)[:, 1])
print(f"Tuned XGBoost (full data) AUC: {auc_tuned:.4f}")

joblib.dump(xgb_tuned, "/content/drive/MyDrive/SmartRail/xgboost_tuned.pkl")

with open("/content/drive/MyDrive/SmartRail/results.json", "r") as f:
    results = json.load(f)
results["XGBoost_tuned"] = {"auc": auc_tuned, "params": best_params}
with open("/content/drive/MyDrive/SmartRail/results.json", "w") as f:
    json.dump(results, f)

Tuned XGBoost (full data) AUC: 1.0000


In [ ]:
log_reg = joblib.load("/content/drive/MyDrive/SmartRail/log_reg.pkl")
rf = joblib.load("/content/drive/MyDrive/SmartRail/random_forest.pkl")
scaler2 = joblib.load("/content/drive/MyDrive/SmartRail/scaler2.pkl")

print("Models loaded")

Models loaded


In [ ]:
X_test_time_scaled = scaler2.transform(X_test_time)

joblib.dump(xgb_tuned, "/content/drive/MyDrive/SmartRail/xgboost_tuned_final.pkl")

with open("/content/drive/MyDrive/SmartRail/results.json", "r") as f:
    results = json.load(f)

print("All results so far:")
for k, v in results.items():
    print(k, "->", v if not isinstance(v, dict) or "params" not in v else {"auc": v.get("auc")})

print("\nSaved — safe to close now")

All results so far:
Logistic Regression -> {'precision': 0.57, 'recall': 1.0, 'f1': 0.73}
Naive Bayes -> {'precision': 0.47, 'recall': 0.99, 'f1': 0.63, 'auc': 0.9859}
KNN -> {'precision': 0.96, 'recall': 0.96, 'f1': 0.96, 'auc': 0.9971}
SVM -> {'precision': 0.76, 'recall': 0.99, 'f1': 0.86, 'auc': 0.9993}
Decision Tree -> {'precision': 0.86, 'recall': 1.0, 'f1': 0.92, 'auc': 0.9991}
Bagging -> {'auc': 0.9999}
Random Forest -> {'auc': 1.0}
AdaBoost -> {'auc': 0.9999}
Gradient Boosting -> {'auc': 1.0}
XGBoost -> {'auc': 1.0}
XGBoost_final_timesplit -> {'auc': 0.9997}
XGBoost_tuned -> {'auc': 0.9999679908500649}

Saved — safe to close now
